# Modul 2: Backpropagation dan Automatic Differentiation

**Nama:** Fathya Intami Gusda
**NIM:** 123450095  
**Kelas:** RB  
**Tanggal:** 2026-09-22  

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [ ]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

NIM = '123450095'                     # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42   # seed individual
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
print({'python': platform.python_version(), 'numpy': np.__version__,
       'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

{'python': '3.13.15', 'numpy': '2.1.3', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 95}


## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Gradien lokal vs gradien total pada satu simpul:** Gradien lokal adalah turunan suatu keluaran terhadap masukan pada satu operasi/simpul. Gradien total adalah pengaruh masukan terhadap loss setelah memperhitungkan seluruh jalur perhitungan dari simpul tersebut ke loss menggunakan aturan rantai (chain rule).
2. **Mengapa `backward()` hanya dapat dipanggil pada tensor skalar:** Karena backward() menghitung turunan loss terhadap parameter. Pada tensor skalar, turunannya memiliki satu nilai sehingga dapat langsung menjadi titik awal (seed) backpropagation. Jika output bukan skalar, diperlukan gradien awal (upstream gradient) untuk setiap elemennya.
3. **Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`:** Gradien akan terakumulasi, bukan diganti. Jadi jika backward() dipanggil dua kali dengan kondisi yang sama, nilai .grad menjadi dua kali gradien dari satu backward. Oleh karena itu, zero_grad() digunakan untuk mengosongkan gradien sebelum iterasi berikutnya.
4. **Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$, bukan $-y/p$:** Karena `BCEWithLogitsLoss` menggabungkan fungsi sigmoid dan binary cross-entropy secara langsung. Jika \(p=\sigma(z)\), penerapan *chain rule* terhadap logit \(z\) menghasilkan turunan yang disederhanakan menjadi \(p-y\). Sementara itu, bentuk \(-y/p\) merupakan bagian dari turunan binary cross-entropy terhadap probabilitas \(p\), bukan turunan loss terhadap logit \(z\).

**Graf komputasi Kasus 1.** Tuliskan urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$, lalu tandai gradien lokal di setiap simpul:
**Graf komputasi Kasus 1:** Urutan simpulNYA adalah:

$$
\mathbf{x}
\rightarrow
\mathbf{z}^{(1)}=\mathbf{x}W^{(1)}+\mathbf{b}^{(1)}
\rightarrow
\mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)})
\rightarrow
z^{(2)}=\mathbf{h}W^{(2)}+b^{(2)}
\rightarrow
p=\sigma(z^{(2)})
\rightarrow
\mathcal{L}=\operatorname{BCE}(p,y)
$$



## B. Turunan manual - 20 poin

Kasus 1 memakai nilai tetap berikut:

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Tuliskan penurunan Anda **berurutan** di sini, satu baris satu langkah:

1. $\partial\mathcal{L}/\partial z^{(2)} = p - y = 0.9241418 - 1 = -0.075858$
2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} = (\partial\mathcal{L}/\partial z^{(2)}) \mathbf{h}^T = -0.075858 \begin{bmatrix}1.5 \\ 1.0\end{bmatrix}^T = \begin{bmatrix}-0.113787 & -0.075858\end{bmatrix}$
3. $\partial\mathcal{L}/\partial b^{(2)} = \partial\mathcal{L}/\partial z^{(2)} = -0.075858$
4. $\partial\mathcal{L}/\partial \mathbf{h} = (\mathbf{W}^{(2)})^T (\partial\mathcal{L}/\partial z^{(2)}) = \begin{bmatrix}2 \\ -1\end{bmatrix} (-0.075858) = \begin{bmatrix}-0.151716 \\ 0.075858\end{bmatrix}$
5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} = (\partial\mathcal{L}/\partial \mathbf{h}) \odot \text{ReLU}'(\mathbf{z}^{(1)}) = \begin{bmatrix}-0.151716 \\ 0.075858\end{bmatrix} \odot \begin{bmatrix}1 \\ 1\end{bmatrix} = \begin{bmatrix}-0.151716 \\ 0.075858\end{bmatrix}$
6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} = (\partial\mathcal{L}/\partial \mathbf{z}^{(1)}) \mathbf{x}^T = \begin{bmatrix}-0.151716 \\ 0.075858\end{bmatrix} \begin{bmatrix}2 & -1\end{bmatrix} = \begin{bmatrix}-0.303432 & 0.151716 \\ 0.151716 & -0.075858\end{bmatrix}$
7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} = \partial\mathcal{L}/\partial \mathbf{z}^{(1)} = \begin{bmatrix}-0.151716 \\ 0.075858\end{bmatrix}$

Cantumkan pula shape setiap gradien:
- $\partial L / \partial W^{(2)}$: (2,)
- $\partial L / \partial b^{(2)}$: skalar
- $\partial L / \partial h$: (2,)
- $\partial L / \partial z^{(1)}$: (2,)
- $\partial L / \partial W^{(1)}$: (2, 2)
- $\partial L / \partial b^{(1)}$: (2,)


In [ ]:
x  = np.array([2.0, -1.0])
W1 = np.array([[0.5, -0.5], [1.0, 1.0]])
b1 = np.array([0.0, 0.0])
W2 = np.array([2.0, -1.0])
b2 = 0.5
y  = 1.0

def forward(x, W1, b1, W2, b2, y):
    """TODO 1: kembalikan dict berisi z1, h, z2, p, dan loss."""

    # Forward pass layer pertama
    z1 = x @ W1.T + b1

    # Aktivasi ReLU
    h = np.maximum(0, z1)

    # Forward pass output
    z2 = h @ W2 + b2

    # Sigmoid
    p = 1 / (1 + np.exp(-z2))

    # Binary Cross-Entropy
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))

    return {
        'z1': z1,
        'h': h,
        'z2': z2,
        'p': p,
        'loss': loss
    }

nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

# Pemeriksaan wajib: jangan diubah.
assert np.allclose(nilai['z1'], [1.5, 1.0]), 'z1 belum benar'
assert np.isclose(nilai['z2'], 2.5), 'logit belum benar'
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6), 'loss belum benar'
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [ ]:
def backward(x, W1, W2, y, nilai):
    """TODO 2: kembalikan dict gradien untuk 'W1', 'b1', 'W2', 'b2'.

    Urutan pengerjaan: dz2 -> (dW2, db2, dh) -> dz1 -> (dW1, db1).
    Ingat gradien lokal ReLU dan bentuk perkalian luar untuk dW1.
    """

    # Gradien loss terhadap z2
    dz2 = nilai['p'] - y

    # Gradien layer output
    dW2 = dz2 * nilai['h']
    db2 = dz2
    dh = dz2 * W2

    # Gradien ReLU
    dz1 = dh * (nilai['z1'] > 0)

    # Gradien layer pertama
    dW1 = np.outer(dz1, x)
    db1 = dz1

    return {
        'W1': dW1,
        'b1': db1,
        'W2': dW2,
        'b2': db2
    }


grad_manual = backward(x, W1, W2, y, nilai)

for nama, v in grad_manual.items():
    print(f'{nama:>3}: {np.round(v, 7)}')

# Pemeriksaan wajib: dua angka kunci dari modul.
assert np.allclose(
    grad_manual['W2'],
    [-0.1137873, -0.0758582],
    atol=1e-6
)
assert grad_manual['W1'].shape == W1.shape, 'shape dW1 harus sama dengan W1'

print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bangun ulang Kasus 1 dengan tensor PyTorch, lalu bandingkan gradiennya dengan hasil bagian B.

In [ ]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx, ty = torch.tensor(x), torch.tensor(y)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    """TODO 3: hitung loss dengan BCEWithLogitsLoss pada LOGIT."""

    # Forward pass layer pertama
    tz1 = tx @ tW1.T + tb1

    # ReLU
    th = torch.relu(tz1)

    # Output berupa logit
    tz2 = th @ tW2 + tb2

    # BCEWithLogitsLoss langsung menerima logit
    loss = kriteria(tz2, ty)

    return loss

loss = forward_torch()
loss.backward()

for nama, t in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(t.grad.numpy() - grad_manual[nama]))
    print(f'{nama}: autograd = {np.round(t.grad.numpy(), 7)}   selisih maks = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} belum cocok dengan hasil manual'
print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]   selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582]   selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582]   selisih maks = 0.00e+00
b2: autograd = -0.0758582   selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [ ]:
# TODO 4: panggil backward() sekali lagi TANPA menghapus gradien,
#         cetak tW2.grad, lalu hapus gradien dan hitung ulang.
#         Jelaskan hasilnya pada sel markdown di bawah.

# Panggil backward() kedua kali tanpa zero_grad()
loss = forward_torch()
loss.backward()

print("Gradien tW2 setelah backward() kedua kali:")
print(tW2.grad)

# Hapus gradien
tW1.grad = None
tb1.grad = None
tW2.grad = None
tb2.grad = None

# Hitung ulang forward dan backward
loss = forward_torch()
loss.backward()

print("\nGradien tW2 setelah gradien dihapus lalu backward() lagi:")
print(tW2.grad)

Gradien tW2 setelah backward() kedua kali:
tensor([-0.2276, -0.1517])

Gradien tW2 setelah gradien dihapus lalu backward() lagi:
tensor([-0.1138, -0.0759])


**Penjelasan akumulasi gradien:** Ketika backward() dipanggil dua kali tanpa menghapus gradien sebelumnya, nilai gradien akan terakumulasi sehingga menjadi 2 kali lipat dari gradien satu kali backward(). Hal ini terjadi karena PyTorch secara default menambahkan gradien baru ke nilai yang sudah tersimpan pada .grad. Oleh karena itu, pada training loop biasanya digunakan optimizer.zero_grad() sebelum loss.backward() untuk menghapus gradien dari iterasi sebelumnya, sehingga setiap iterasi menggunakan gradien yang baru dan tidak terjadi akumulasi yang tidak diinginkan.

## D. Gradient checking - 20 poin

Bandingkan gradien analitik dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

Ambang lulus: seluruh baris di bawah $10^{-5}$.

In [ ]:
EPS = 1e-5

def loss_dengan(param, i, delta):
    """TODO 5: salin parameter, geser satu komponen sebesar delta, kembalikan loss."""

    # Salin parameter
    W1_baru = W1.copy()
    b1_baru = b1.copy()
    W2_baru = W2.copy()
    b2_baru = b2

    # Geser satu komponen parameter
    if param == 'W1':
        W1_baru[i] += delta
    elif param == 'b1':
        b1_baru[i] += delta
    elif param == 'W2':
        W2_baru[i] += delta
    elif param == 'b2':
        b2_baru += delta

    # Hitung loss dengan parameter yang sudah digeser
    nilai_baru = forward(
        x, W1_baru, b1_baru, W2_baru, b2_baru, y
    )

    return nilai_baru['loss']


def finite_difference(param, i):
    """TODO 6: kembalikan gradien numerik dengan selisih terpusat."""

    loss_plus = loss_dengan(param, i, EPS)
    loss_minus = loss_dengan(param, i, -EPS)

    numerik = (loss_plus - loss_minus) / (2 * EPS)

    return float(numerik)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.numpy()
    auto = auto[i] if i != () else auto
    numerik = finite_difference(nama, i)
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
print(tabel.to_string(index=False))
print('\nrelative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9, 'tabel harus memuat sembilan komponen parameter'
assert tabel['rel_err'].max() < 1e-5, 'masih ada baris yang melampaui ambang'

parameter indeks      manual    autograd     numerik        rel_err
       W1 (0, 0) -0.30343272 -0.30343272 -0.30343272 1.00453007e-10
       W1 (0, 1)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 0)  0.15171636  0.15171636  0.15171636 2.95622632e-11
       W1 (1, 1) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b1   (0,) -0.15171636 -0.15171636 -0.15171636 2.95622632e-11
       b1   (1,)  0.07585818  0.07585818  0.07585818 5.73360674e-11
       W2   (0,) -0.11378727 -0.11378727 -0.11378727 6.76755052e-11
       W2   (1,) -0.07585818 -0.07585818 -0.07585818 5.73360674e-11
       b2     () -0.07585818 -0.07585818 -0.07585818 5.73360674e-11

relative error maksimum: 1.004530066507852e-10


In [ ]:
# TODO 7: simpan tabel ke M02_NIM_metrics.csv (ganti NIM dengan NIM Anda).
tabel.insert(0, 'run_id', 'gradcheck')
tabel.insert(1, 'seed', SEED)
tabel.to_csv(f'M02_123450095_metrics.csv', index=False)
print('tersimpan')

tersimpan


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [ ]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def train_rusak(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Linear(4, 1),
    )
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)
        loss = kriteria(torch.sigmoid(logits), y_xor)
        opt.step()
        loss.backward()
        riwayat.append(loss.item())
    return model, riwayat

model_rusak, riwayat_rusak = train_rusak()
print(f'loss awal  : {riwayat_rusak[0]:.4f}')
print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_rusak(X_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

loss = kriteria(torch.sigmoid(logits), y_xor)### Temuan kesalahan

Isi tabel berikut. Setiap baris harus menyebut **baris kode**, **alasan**, dan **gejala** yang teramati.

| No | Baris kode bermasalah | Mengapa keliru | Gejala yang terlihat |
|----|----------------------|----------------|----------------------|
| 1  | loss = kriteria(torch.sigmoid(logits), y_xor) | BCEWithLogitsLoss sudah melakukan sigmoid secara internal, sehingga sigmoid tidak perlu dilakukan terlebih dahulu. | Perhitungan loss/gradien menjadi tidak sesuai dan training dapat berjalan kurang baik. |
| 2  | opt.step() sebelum loss.backward() | optimizer.step() membutuhkan gradien yang sudah dihitung oleh backward(). Pada saat step() dipanggil, gradien belum tersedia. | Parameter tidak diperbarui berdasarkan gradien pada iterasi tersebut; training tidak berjalan sebagaimana mestinya. |
| 3  | Tidak ada opt.zero_grad() | Gradien pada .grad secara default diakumulasi setiap kali backward() dipanggil. | Gradien terus menumpuk dari iterasi sebelumnya sehingga update parameter menjadi tidak sesuai. |
| 4  | loss.backward() setelah opt.step() | Urutan training yang benar adalah menghitung gradien dengan backward() terlebih dahulu, kemudian menggunakan gradien tersebut dengan step(). | Update bobot terlambat/tidak menggunakan gradien dari loss pada iterasi saat ini sehingga loss dan prediksi tidak belajar dengan benar. |

In [ ]:
# TODO 8: perbaiki SATU PER SATU.
# Setiap tahap mempertahankan perbaikan sebelumnya dan menambahkan
# satu perbaikan baru.

tahap = []


# TAHAP 1
# Perbaikan: tambahkan aktivasi ReLU di Sequential
def train_perbaikan_1(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)

    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        # Masih menggunakan sigmoid sebelum BCE
        logits = model(X_xor)
        loss = kriteria(torch.sigmoid(logits), y_xor)

        # backward sebelum step agar tidak terjadi
        # inplace modification pada computational graph
        loss.backward()
        opt.step()

        # Belum menggunakan zero_grad()
        # sesuai tahap perbaikan berikutnya
        for p in model.parameters():
            p.grad = None

        riwayat.append(loss.item())

    return model, riwayat


model, riwayat = train_perbaikan_1()

with torch.no_grad():
    pred = (torch.sigmoid(model(X_xor)) > 0.5).int().flatten()
    benar = int((pred == y_xor.int().flatten()).sum())

tahap.append({
    'tahap': 'perbaikan-1',
    'yang_diperbaiki': 'aktivasi ReLU di Sequential',
    'loss_awal': riwayat[0],
    'loss_akhir': riwayat[-1],
    'benar': benar
})


# TAHAP 2
# Perbaikan: hilangkan sigmoid sebelum BCEWithLogitsLoss
def train_perbaikan_2(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)

    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)

        # Perbaikan 2
        loss = kriteria(logits, y_xor)

        loss.backward()
        opt.step()

        # Belum menggunakan optimizer.zero_grad()
        for p in model.parameters():
            p.grad = None

        riwayat.append(loss.item())

    return model, riwayat


model, riwayat = train_perbaikan_2()

with torch.no_grad():
    pred = (torch.sigmoid(model(X_xor)) > 0.5).int().flatten()
    benar = int((pred == y_xor.int().flatten()).sum())

tahap.append({
    'tahap': 'perbaikan-2',
    'yang_diperbaiki': 'hilangkan sigmoid sebelum BCE',
    'loss_awal': riwayat[0],
    'loss_akhir': riwayat[-1],
    'benar': benar
})


# TAHAP 3
# Perbaikan: opt.step() setelah backward()
def train_perbaikan_3(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)

    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)
        loss = kriteria(logits, y_xor)

        # Perbaikan 3
        loss.backward()
        opt.step()

        # Gradien dihapus secara manual karena
        # zero_grad() baru diperkenalkan pada tahap 4
        for p in model.parameters():
            p.grad = None

        riwayat.append(loss.item())

    return model, riwayat


model, riwayat = train_perbaikan_3()

with torch.no_grad():
    pred = (torch.sigmoid(model(X_xor)) > 0.5).int().flatten()
    benar = int((pred == y_xor.int().flatten()).sum())

tahap.append({
    'tahap': 'perbaikan-3',
    'yang_diperbaiki': 'opt.step() setelah backward()',
    'loss_awal': riwayat[0],
    'loss_akhir': riwayat[-1],
    'benar': benar
})


# TAHAP 4
# Perbaikan: gunakan optimizer.zero_grad()
def train_perbaikan_4(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)

    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )

    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        # Perbaikan 4
        opt.zero_grad()

        logits = model(X_xor)
        loss = kriteria(logits, y_xor)

        loss.backward()
        opt.step()

        riwayat.append(loss.item())

    return model, riwayat


model, riwayat = train_perbaikan_4()

with torch.no_grad():
    pred = (torch.sigmoid(model(X_xor)) > 0.5).int().flatten()
    benar = int((pred == y_xor.int().flatten()).sum())

tahap.append({
    'tahap': 'perbaikan-4',
    'yang_diperbaiki': 'opt.zero_grad()',
    'loss_awal': riwayat[0],
    'loss_akhir': riwayat[-1],
    'benar': benar
})


# Tampilkan hasil semua tahap
tabel_tahap = pd.DataFrame(tahap)
print(tabel_tahap.to_string(index=False))

      tahap               yang_diperbaiki  loss_awal  loss_akhir  benar
perbaikan-1   aktivasi ReLU di Sequential 0.71781101  0.69626977      2
perbaikan-2 hilangkan sigmoid sebelum BCE 0.69845659  0.22002037      4
perbaikan-3 opt.step() setelah backward() 0.69845659  0.22002037      4
perbaikan-4               opt.zero_grad() 0.69845659  0.22002037      4


In [ ]:
def train_benar(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        opt.zero_grad()
        logits = model(X_xor)
        loss = kriteria(logits, y_xor)
        loss.backward()
        opt.step()
        riwayat.append(loss.item())
    return model, riwayat

model_benar, riwayat_benar = train_benar()
print(f'loss awal  : {riwayat_benar[0]:.4f}')
print(f'loss akhir : {riwayat_benar[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_benar(X_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())


loss awal  : 0.6985
loss akhir : 0.2200
prediksi   : [0, 1, 1, 0]
target     : [0, 1, 1, 0]


In [ ]:
# TODO 10: gabungkan catatan tahap perbaikan ke metrics.csv.
df_tahap = pd.DataFrame(tahap)
df_tahap.insert(0, 'seed', SEED)
df_tahap.to_csv(f'M02_123450095_metrics_loop.csv', index=False)
print(df_tahap.to_string(index=False))

 seed       tahap               yang_diperbaiki  loss_awal  loss_akhir  benar
   95 perbaikan-1   aktivasi ReLU di Sequential 0.71781101  0.69626977      2
   95 perbaikan-2 hilangkan sigmoid sebelum BCE 0.69845659  0.22002037      4
   95 perbaikan-3 opt.step() setelah backward() 0.69845659  0.22002037      4
   95 perbaikan-4               opt.zero_grad() 0.69845659  0.22002037      4


## F. Tugas individu

Kerjakan ketiganya di sel-sel baru di bawah bagian ini.

1. **Perluasan jaringan.** Tambahkan neuron ketiga pada hidden layer: baris $[-1\;\;0.5]$ pada $\mathbf{W}^{(1)}$, bias $0{,}25$, dan komponen $-0{,}5$ pada $\mathbf{W}^{(2)}$. Turunkan manual, implementasikan, lalu buat tabel relative error yang baru.
2. **Batch dua contoh.** Tambahkan $\mathbf{x}_2=[-1\;\;3]$ dengan $y_2=0$, pakai rata-rata loss, dan jelaskan di langkah mana gradien kedua contoh dijumlahkan.
3. **Laporan diagnosis.** Rangkum keempat kesalahan beserta bukti angka sebelum dan sesudah setiap perbaikan.

## G. Pertanyaan analisis

1. Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar? TODO
2. Apa yang terjadi pada tabel bila $\epsilon = 10^{-9}$? Jalankan dan jelaskan. TODO
3. Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch? TODO
4. Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka? TODO
5. Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat) TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Turunan manual ditulis pada sel markdown bagian B.
- [ ] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [ ] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [ ] Notebook lolos *Restart Kernel and Run All*.
- [ ] Berkas: `M02_NIM.ipynb`, `M02_NIM.pdf`, `M02_NIM_metrics.csv`.